# 6장 · 우연은 시뮬레이션으로: 표집분포

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [6장 · 우연은 시뮬레이션으로: 표집분포](https://grow.minds.kr/textbooks/css-methods/causal/book/ch06-우연은-시뮬레이션으로.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 표집분포를 직접 생성
566명을 모집단 삼아 50명씩 1,000번 뽑아 평균을 모읍니다.

In [ ]:
svy = load("svy"); pop = svy.mil.values
rng = np.random.default_rng(73)
means = np.array([rng.choice(pop, 50, replace=False).mean() for _ in range(1000)])
print("첫 세 표본:", [round(x,2) for x in means[:3]])         # 4.82, 5.14, 4.69
print("표집분포 표준편차(표준오차):", round(means.std(ddof=1), 3))   # 0.175

## 3. 공식은 시뮬레이션의 요약
SD÷√n 이 시뮬레이션 값과 사실상 같습니다.

In [ ]:
print("공식 SD/√n =", round(pop.std(ddof=1)/np.sqrt(50), 3))     # 0.177

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 *검증*이 이 책의 핵심입니다.